In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os, json
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, matthews_corrcoef, roc_auc_score, confusion_matrix)


majority_ensemble


In [ ]:
OUTDIR = "/content/drive/MyDrive/NEW/EnsembleResults"
os.makedirs(OUTDIR, exist_ok=True)

In [ ]:
# -------- Path resolver: add candidates for each model --------
CANDIDATES = {
    "llama-3.2-3B-instruct": [
        "/content/drive/MyDrive/NEW/BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct/test_predictions_with_probs.csv",
    ],
    "alpaca-orca": [
        "/content/drive/MyDrive/NEW/BanglaLLama-3.2-3b-bangla-alpaca-orca-instruct-v0.0.1/test_predictions_with_probs.csv",
    ],
    "culturax-3b-seqcls-new": [
        "/content/drive/MyDrive/NEW/culturax-base-3b-seqcls-new/test_predictions_with_probs.csv",
    ],
    "qwen-2.5-3B-instruct": [
        "/content/drive/MyDrive/NEW/Bangla-s1k-qwen-2.5-3B-Instruct/test_predictions_with_probs.csv",
    ],
}


In [ ]:
def pick_first_existing(path_list):
    for p in path_list:
        if os.path.exists(p): return p
    return None

FILES = []
for tag, options in CANDIDATES.items():
    found = pick_first_existing(options)
    if not found:
        raise FileNotFoundError(f"❌ Missing CSV for '{tag}'. Tried:\n  - " + "\n  - ".join(options))
    FILES.append(found)

print("✅ Using CSVs:")
for p in FILES: print(" -", p)

✅ Using CSVs:
 - /content/drive/MyDrive/NEW/BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct/test_predictions_with_probs.csv
 - /content/drive/MyDrive/NEW/BanglaLLama-3.2-3b-bangla-alpaca-orca-instruct-v0.0.1/test_predictions_with_probs.csv
 - /content/drive/MyDrive/NEW/culturax-base-3b-seqcls-new/test_predictions_with_probs.csv
 - /content/drive/MyDrive/NEW/Bangla-s1k-qwen-2.5-3B-Instruct/test_predictions_with_probs.csv


In [ ]:
# -------- Load & align --------
P0, P1 = "Prob_Fake(0)", "Prob_NonFake(1)"  # class 1 = Non-Fake
def load_one(path):
    df = pd.read_csv(path)
    need = ["Review","True",P0,P1]
    miss = [c for c in need if c not in df.columns]
    if miss: raise ValueError(f"{path} missing {miss}")
    df = df[["Review","True",P0,P1]].copy()
    df.columns = ["review","true","p0","p1"]
    return df

frames = [load_one(p) for p in FILES]
base = frames[0].copy()
for f in frames[1:]:
    if not (len(base)==len(f) and (base["review"]==f["review"]).all() and (base["true"]==f["true"]).all()):
        base = base.merge(f, on=["review","true"], suffixes=("","_m"))
    else:
        base[f"p0_{id(f)}"] = f["p0"].values
        base[f"p1_{id(f)}"] = f["p1"].values

p1_cols = [c for c in base.columns if c.startswith("p1")]
if "p1" in base.columns: p1_cols = ["p1"] + [c for c in p1_cols if c!="p1"]

y_true = base["true"].astype(int).values
prob1  = np.vstack([base[c].values for c in p1_cols])  # (M, N)


In [ ]:
# -------- Majority vote --------
def majority_vote(prob, thr=0.5):
    votes = (prob >= thr).astype(int)
    return (votes.sum(axis=0) >= (prob.shape[0]//2 + 1)).astype(int)

maj_pred = majority_vote(prob1, 0.5)
maj_prob = prob1.mean(axis=0)  # for ROC etc.

def metrics(y, prob, thr=0.5):
    yhat = (prob >= thr).astype(int)
    rep = classification_report(y, yhat, target_names=["Fake(0)","Non-Fake(1)"], digits=4, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y,yhat)),
        "precision": float(precision_recall_fscore_support(y,yhat,average="binary",zero_division=0)[0]),
        "recall": float(precision_recall_fscore_support(y,yhat,average="binary",zero_division=0)[1]),
        "f1_binary": float(f1_score(y,yhat)),
        "f1_macro": float(f1_score(y,yhat,average="macro")),
        "mcc": float(matthews_corrcoef(y,yhat)),
        "roc_auc": float(roc_auc_score(y, prob)) if len(np.unique(y))==2 else float("nan"),
        "confusion_matrix": confusion_matrix(y,yhat).tolist(),
        "report": rep, "threshold": 0.5
    }

m = metrics(y_true, maj_prob, 0.5)

# -------- Save outputs + artifact --------
out = frames[0][["review","true"]].copy()
out["maj_prob_nonfake"] = maj_prob
out["maj_pred"] = maj_pred
out.to_csv(os.path.join(OUTDIR, "majority_ensemble_outputs.csv"), index=False, encoding="utf-8")
with open(os.path.join(OUTDIR, "majority_ensemble_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(m, f, indent=2, ensure_ascii=False)
with open(os.path.join(OUTDIR, "majority_artifact.json"), "w", encoding="utf-8") as f:
    json.dump({"method":"majority","threshold":0.5}, f, indent=2)

print("=== Majority vote ===\n", m["report"])
print("➡ Saved to", OUTDIR)

=== Majority vote ===
               precision    recall  f1-score   support

     Fake(0)     0.9865    0.9546    0.9703      1146
 Non-Fake(1)     0.9670    0.9903    0.9785      1539

    accuracy                         0.9750      2685
   macro avg     0.9767    0.9724    0.9744      2685
weighted avg     0.9753    0.9750    0.9750      2685

➡ Saved to /content/drive/MyDrive/NEW/EnsembleResults


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import joblib

In [ ]:
# ---- helper re-use ----
def tune_threshold(y, prob, metric="f1"):
    best_t, best_s = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 19):
        yhat = (prob >= t).astype(int)
        s = f1_score(y, yhat) if metric == "f1" else matthews_corrcoef(y, yhat)
        if s > best_s:
            best_s, best_t = s, float(t)
    return best_t

def full_metrics(y, prob, thr):
    yhat = (prob >= thr).astype(int)
    rep = classification_report(y, yhat, target_names=["Fake(0)", "Non-Fake(1)"], digits=4, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y, yhat)),
        "precision": float(precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)[0]),
        "recall": float(precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)[1]),
        "f1_binary": float(f1_score(y, yhat)),
        "f1_macro": float(f1_score(y, yhat, average="macro")),
        "mcc": float(matthews_corrcoef(y, yhat)),
        "roc_auc": float(roc_auc_score(y, prob)) if len(np.unique(y)) == 2 else float("nan"),
        "threshold": float(thr),
        "report": rep,
    }

summary_rows = []
# We already saved Majority above; add its headline metrics to the comparison
summary_rows.append({
    "method": "majority",
    "accuracy": m["accuracy"], "f1_binary": m["f1_binary"],
    "f1_macro": m["f1_macro"], "mcc": m["mcc"], "roc_auc": m["roc_auc"],
    "threshold": m["threshold"]
})


soft_ensemble

In [ ]:
soft_prob = prob1.mean(axis=0)
soft_thr = tune_threshold(y_true, soft_prob, metric="f1")
soft_m = full_metrics(y_true, soft_prob, soft_thr)


In [ ]:
# Save outputs/artifacts
soft_out = frames[0][["review","true"]].copy()
soft_out["soft_prob_nonfake"] = soft_prob
soft_out["soft_pred"] = (soft_prob >= soft_thr).astype(int)
soft_out.to_csv(os.path.join(OUTDIR, "soft_ensemble_outputs.csv"), index=False, encoding="utf-8")
with open(os.path.join(OUTDIR, "soft_ensemble_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(soft_m, f, indent=2, ensure_ascii=False)
with open(os.path.join(OUTDIR, "soft_artifact.json"), "w", encoding="utf-8") as f:
    json.dump({"method": "soft", "threshold": float(soft_thr)}, f, indent=2)

print(f"\n=== Soft vote (thr={soft_thr:.3f}) ===\n", soft_m["report"])
summary_rows.append({"method":"soft", **{k:soft_m[k] for k in ["accuracy","f1_binary","f1_macro","mcc","roc_auc","threshold"]}})


=== Soft vote (thr=0.600) ===
               precision    recall  f1-score   support

     Fake(0)     0.9822    0.9616    0.9718      1146
 Non-Fake(1)     0.9718    0.9870    0.9794      1539

    accuracy                         0.9762      2685
   macro avg     0.9770    0.9743    0.9756      2685
weighted avg     0.9763    0.9762    0.9761      2685



WEIGHTED SOFT

In [ ]:
# read weights from each model's metrics.json (fallback=1.0)
def weight_from_metrics(csv_path):
    mdir = os.path.dirname(csv_path)
    mj = os.path.join(mdir, "metrics.json")
    if os.path.isfile(mj):
        try:
            return float(json.load(open(mj, "r", encoding="utf-8")).get("f1_weighted", 1.0))
        except Exception:
            return 1.0
    return 1.0

weights = np.array([weight_from_metrics(p) for p in FILES], dtype=float)
if weights.sum() == 0: weights = np.ones_like(weights)
weights = weights / weights.sum()

w_prob = (weights[:, None] * prob1).sum(axis=0)
w_thr = tune_threshold(y_true, w_prob, metric="f1")
w_m = full_metrics(y_true, w_prob, w_thr)

In [ ]:
# Save outputs/artifacts
w_out = frames[0][["review","true"]].copy()
w_out["weighted_prob_nonfake"] = w_prob
w_out["weighted_pred"] = (w_prob >= w_thr).astype(int)
w_out.to_csv(os.path.join(OUTDIR, "weighted_ensemble_outputs.csv"), index=False, encoding="utf-8")
with open(os.path.join(OUTDIR, "weighted_ensemble_metrics.json"), "w", encoding="utf-8") as f:
    json.dump({**w_m, "weights": [float(x) for x in weights]}, f, indent=2, ensure_ascii=False)
with open(os.path.join(OUTDIR, "weighted_artifact.json"), "w", encoding="utf-8") as f:
    json.dump({"method":"weighted", "threshold": float(w_thr), "weights": [float(x) for x in weights]}, f, indent=2)

print(f"\n=== Weighted soft (thr={w_thr:.3f}, weights={weights.round(3).tolist()}) ===\n", w_m["report"])
summary_rows.append({"method":"weighted", **{k:w_m[k] for k in ["accuracy","f1_binary","f1_macro","mcc","roc_auc","threshold"]}})




=== Weighted soft (thr=0.600, weights=[0.25, 0.25, 0.25, 0.25]) ===
               precision    recall  f1-score   support

     Fake(0)     0.9822    0.9616    0.9718      1146
 Non-Fake(1)     0.9718    0.9870    0.9794      1539

    accuracy                         0.9762      2685
   macro avg     0.9770    0.9743    0.9756      2685
weighted avg     0.9763    0.9762    0.9761      2685



 STACKING (Logistic Regression)

In [ ]:
# 5-fold OOF to choose a robust threshold (avoid leakage)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros_like(y_true, dtype=float); fold_thrs = []
for tr, va in skf.split(prob1.T, y_true):
    Xtr, Xva = prob1[:, tr].T, prob1[:, va].T
    ytr, yva = y_true[tr], y_true[va]
    clf = LogisticRegression(max_iter=200)
    clf.fit(Xtr, ytr)
    pva = clf.predict_proba(Xva)[:, 1]
    oof[va] = pva
    fold_thrs.append(tune_threshold(yva, pva, metric="f1"))

stack_thr = float(np.mean(fold_thrs))
stack_oof_m = full_metrics(y_true, oof, stack_thr)
print(f"\n=== Stacking (OOF, thr={stack_thr:.3f}) ===\n", stack_oof_m["report"])



=== Stacking (OOF, thr=0.430) ===
               precision    recall  f1-score   support

     Fake(0)     0.9838    0.9546    0.9690      1146
 Non-Fake(1)     0.9669    0.9883    0.9775      1539

    accuracy                         0.9739      2685
   macro avg     0.9754    0.9715    0.9733      2685
weighted avg     0.9741    0.9739    0.9739      2685



In [ ]:
# Fit final meta-model on full meta-features and save for future inference
X_full = prob1.T
stack_clf = LogisticRegression(max_iter=200)
stack_clf.fit(X_full, y_true)
joblib.dump(stack_clf, os.path.join(OUTDIR, "stacking_model.joblib"))
with open(os.path.join(OUTDIR, "stacking_artifact.json"), "w", encoding="utf-8") as f:
    json.dump({"method":"stacking", "threshold": stack_thr, "n_meta_features": int(prob1.shape[0])}, f, indent=2)


In [ ]:
# Save full-fit predictions and metrics
stack_prob = stack_clf.predict_proba(X_full)[:, 1]
stack_m = full_metrics(y_true, stack_prob, stack_thr)
stack_out = frames[0][["review","true"]].copy()
stack_out["stack_prob_nonfake"] = stack_prob
stack_out["stack_pred"] = (stack_prob >= stack_thr).astype(int)
stack_out.to_csv(os.path.join(OUTDIR, "stacking_ensemble_outputs.csv"), index=False, encoding="utf-8")
with open(os.path.join(OUTDIR, "stacking_ensemble_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(stack_m, f, indent=2, ensure_ascii=False)

summary_rows.append({"method":"stacking", **{k:stack_m[k] for k in ["accuracy","f1_binary","f1_macro","mcc","roc_auc","threshold"]}})


In [ ]:
# -------------------- Comparison table + chart --------------------
cmp_df = pd.DataFrame(summary_rows).set_index("method").round(4)
cmp_csv = os.path.join(OUTDIR, "ensemble_comparison.csv")
cmp_df.to_csv(cmp_csv, encoding="utf-8")
print("\n=== Comparison (saved) ===\n", cmp_df)

# Bar chart for F1 (binary) and MCC
plt.figure(figsize=(7.5,4.5))
ax = cmp_df[["f1_binary","mcc"]].plot(kind="bar")
ax.set_xlabel("Method")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.0)
plt.title("Ensemble Methods: F1 (binary) & MCC")
plt.tight_layout()
fig_path = os.path.join(OUTDIR, "ensemble_comparison.png")
plt.savefig(fig_path, dpi=160)
plt.close()
print("📁 Saved:")
print(" -", cmp_csv)
print(" -", fig_path)


=== Comparison (saved) ===
           accuracy  f1_binary  f1_macro     mcc  roc_auc  threshold
method                                                             
majority    0.9750     0.9785    0.9744  0.9492   0.9952       0.50
soft        0.9762     0.9794    0.9756  0.9513   0.9952       0.60
weighted    0.9762     0.9794    0.9756  0.9513   0.9952       0.60
stacking    0.9754     0.9788    0.9748  0.9499   0.9952       0.43
📁 Saved:
 - /content/drive/MyDrive/NEW/EnsembleResults/ensemble_comparison.csv
 - /content/drive/MyDrive/NEW/EnsembleResults/ensemble_comparison.png


<Figure size 750x450 with 0 Axes>